# 10. NSW — 가까운 점끼리 친구 맺기, 친구를 따라 가까이

슬라이드 「HNSW 원리 1·2」를 2차원 점 300개로 직접 만들어 본다.
1. 점을 하나씩 넣으면서 **이미 있는 점 중 가장 가까운 M개**와 양방향 친구를 맺는다
2. 친구가 너무 많아지면 **가장 먼 친구를 끊는다** (가지치기)
3. 검색은 진입점에서 **더 가까운 친구로 이동**하다가 더 못 가면 멈춘다 (그리디)

In [ ]:
from labutil import *
import heapq

X, label = make_clusters(300, 2, n_clusters=6, spread=0.8, center_scale=4, seed=7)
dist = lambda a, b: float(np.linalg.norm(a - b))


def build_nsw(X, M=4, M_max=None):
    """점을 순서대로 추가하며 그래프를 키운다. 반환: 점마다 친구 목록."""
    M_max = M_max or 2 * M
    nbrs = [[] for _ in range(len(X))]
    for i in range(1, len(X)):
        d = np.linalg.norm(X[:i] - X[i], axis=1)
        for j in np.argsort(d)[:M]:                # 이미 있는 점 중 가장 가까운 M개
            nbrs[i].append(j); nbrs[j].append(i)   # 양방향 친구
            if len(nbrs[j]) > M_max:               # 가지치기: j 의 친구 중 가장 먼 친구를 끊는다
                far = max(nbrs[j], key=lambda t: dist(X[t], X[j]))
                nbrs[j].remove(far)                # (HNSW 처럼 j 쪽 목록에서만 뺀다)
    return nbrs


def draw(ax, X, nbrs, upto=None, title=""):
    upto = upto or len(X)
    for i in range(upto):
        for j in nbrs[i]:
            if j < upto and i < j:
                ax.plot(*X[[i, j]].T, color="#9aa5a8", lw=0.6, zorder=1)
    ax.scatter(*X[:upto].T, s=12, color="#3D484A", zorder=2)
    ax.set_title(title); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])

## 1. 증분 구축 — 그래프가 자라는 모습

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, n in zip(axes, [4, 30, 100, 300]):
    draw(ax, X[:n], build_nsw(X[:n], M=4), title=f"{n}개 추가 후 (M=4)")
plt.tight_layout(); plt.show()

**관찰:** 처음 몇 개 점은 서로 멀어도 "가장 가까운 M개"라서 연결된다.
점이 촘촘해진 뒤 들어온 점은 가까운 이웃하고만 연결된다.
→ **먼저 들어온 점이 긴 간선(지름길)**을 갖게 된다. 정말 그런지 재 보자.

In [ ]:
G = build_nsw(X, M=4)
avg_len = [np.mean([dist(X[i], X[j]) for j in G[i]]) for i in range(len(X))]
plt.figure(figsize=(7, 3))
plt.scatter(range(len(X)), avg_len, s=8)
plt.xlabel("추가된 순서"); plt.ylabel("친구까지 평균 거리"); plt.title("먼저 들어온 점일수록 긴 간선")
plt.show()
print(f"처음 30개 평균 {np.mean(avg_len[:30]):.2f}  /  마지막 100개 평균 {np.mean(avg_len[-100:]):.2f}")

## 2. 그리디 탐색 — 친구를 따라 가까이

In [ ]:
def greedy(nbrs, X, q, entry=0):
    """더 가까운 친구로 이동, 더 없으면 멈춤. 반환: (도착점, 경로, 거리계산 횟수)."""
    cur, d_cur, path, n_dist = entry, dist(X[entry], q), [entry], 1
    while True:
        cand = nbrs[cur]
        n_dist += len(cand)
        d = [dist(X[c], q) for c in cand]
        if not cand or min(d) >= d_cur:
            return cur, path, n_dist
        cur, d_cur = cand[int(np.argmin(d))], min(d)
        path.append(cur)


# 질의는 데이터 근처에서 뽑는다 (데이터 점 + 약간의 잡음)
rng = np.random.default_rng(1)
Qs = (X[rng.integers(0, len(X), 500)] + rng.normal(0, 0.5, (500, 2))).astype(np.float32)
truth = brute_knn(X, Qs, 1)[:, 0]

res = [greedy(G, X, qq) for qq in Qs]
fail = [i for i, (r, _, _) in enumerate(res) if r != truth[i]]
print(f"질의 500개: 최근접을 맞힌 비율 {1 - len(fail) / 500:.1%}, "
      f"평균 거리계산 {np.mean([r[2] for r in res]):.0f}회 (전수 비교는 300회)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ok = next(i for i in range(500) if i not in fail and len(res[i][1]) >= 4)   # 몇 걸음 이동한 성공 사례
bad = max(fail, key=lambda i: len(res[i][1]))                              # 가장 멀리 가다 멈춘 실패 사례
for ax, i, name in [(axes[0], ok, "성공"), (axes[1], bad, "실패: 지역 최적")]:
    draw(ax, X, G, title=name)
    _, path, _ = res[i]
    ax.plot(*X[path].T, "-o", color="#FF7043", ms=5, lw=2, zorder=3, label="탐색 경로")
    ax.scatter(*Qs[i], marker="*", s=250, color="#009CA6", zorder=4, label="질의 Q")
    ax.scatter(*X[truth[i]], s=120, facecolors="none", edgecolors="#009CA6", lw=2, zorder=4, label="진짜 최근접")
    ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

**관찰:** 실패한 경우, 멈춘 점의 친구들은 모두 Q 에서 멀지만 그 너머에 진짜 최근접이 있다.
슬라이드의 "운 나쁘면 지역 최적에 빠져 진짜 최근접을 놓친다"가 바로 이것.

## 3. 해결: 후보를 넓게 본다 (ef)
한 점만 들고 다니지 말고 **가장 가까운 후보 ef 개**를 들고 다닌다.
후보 중 아직 안 펼친 점의 친구를 계속 확인하다가, 남은 후보가 모두 현재 ef 개보다 멀면 멈춘다.
(HNSW 의 `ef_search` 가 이것이다. ef=1 이면 위의 그리디와 같다)

In [ ]:
def beam(nbrs, X, q, entry=0, ef=8):
    d0 = dist(X[entry], q)
    visited, n_dist = {entry}, 1
    cand = [(d0, entry)]          # 펼칠 후보 (가까운 순)
    best = [(-d0, entry)]         # 지금까지 찾은 가장 가까운 ef 개 (먼 순 힙)
    while cand:
        d, c = heapq.heappop(cand)
        if d > -best[0][0]:       # 남은 후보가 모두 결과보다 멀면 종료
            break
        for nb in nbrs[c]:
            if nb in visited:
                continue
            visited.add(nb); n_dist += 1
            dn = dist(X[nb], q)
            if len(best) < ef or dn < -best[0][0]:
                heapq.heappush(cand, (dn, nb)); heapq.heappush(best, (-dn, nb))
                if len(best) > ef:
                    heapq.heappop(best)
    return [i for _, i in sorted((-d, i) for d, i in best)], n_dist


def reachable(nbrs, entry=0):
    """진입점에서 친구를 따라 갈 수 있는 점의 비율"""
    seen, stack = {entry}, [entry]
    while stack:
        for nb in nbrs[stack.pop()]:
            if nb not in seen:
                seen.add(nb); stack.append(nb)
    return len(seen) / len(nbrs)


rows = []
for M in [2, 4, 6]:
    g = build_nsw(X, M=M)
    for ef in [1, 4, 16]:
        r = [beam(g, X, qq, ef=ef) for qq in Qs]
        rows.append({"M": M, "ef": ef, "도달가능": reachable(g),
                     "recall@1": np.mean([x[0][0] == t for x, t in zip(r, truth)]),
                     "평균 거리계산": np.mean([x[1] for x in r])})
t = pd.DataFrame(rows)
print("진입점에서 도달 가능한 점의 비율:", t.groupby("M")["도달가능"].first().round(2).to_dict())
t.pivot(index="M", columns="ef", values=["recall@1", "평균 거리계산"]).round(2)

**관찰:**
- `ef` 를 키우면 recall 이 올라가는 대신 거리 계산이 늘어난다 (**정확도 ↔ 속도**)
- `M` 이 너무 작으면 군집끼리 이어지지 않아 **섬**이 생긴다. ef 를 아무리 키워도 섬 밖의 정답은 못 찾는다
- `M` 을 키우면 지역 최적에서 빠져나오기 쉽지만, 한 걸음마다 확인할 친구도 많아진다
- 점 300개에서는 전수 비교(300회)와 차이가 크지 않다. **점이 많아지면** 어떻게 되는지는 11번에서 본다

> 참고: 실제 HNSW 는 가지치기를 "가장 먼 친구"가 아니라 **다양성 휴리스틱**으로 한다.
> 이미 친구인 점과 가까운 후보는 건너뛰어서, 여러 방향으로 친구를 고르게 둔다. 11번에서 비교한다.